# 15. Real Animals: Results

1. Cohort overview: assignment strip + consensus
2. Parameter distributions across cohort
3. GS vs SBI consistency
4. Example animal deep-dives
5. Dynamic SBI trajectories
6. Drill-down (pick any animal)

In [ ]:
%matplotlib inline
from shared_setup import *
import pickle, torch

from plotting.animal_report import (
    plot_animal_summary, plot_cv_results,
    plot_model_fits, plot_sbi_diagnostics,
    _get_sbi_samples,
)
from analysis.consensus import load_all_assignments, consensus_summary
from behav_utils.plotting.styles import COLOURS

BE_COL, SC_COL = COLOURS["BE"], COLOURS["SC"]

In [ ]:
# ── Animals to show in detail (change as needed) ──────────────────────────
EXAMPLE_ANIMALS = ["SS05", "SS07"]

experiment, info = load_data()
snpe = load_snpe_networks()

---
## 1. Assignment Strip

All animals × all methods. Solid colour = significant (p < 0.05).
Faded = direction but not significant. Last column = consensus.

In [ ]:
assign_df = load_all_assignments(RESULTS_DIR, experiment)
print(consensus_summary(assign_df))

In [ ]:
method_cols = ["GS-UM", "GS-CP", "SBI-UM", "SBI-CP"]
all_cols = method_cols + ["Consensus"]
n_methods = len(all_cols)
n_animals = len(assign_df)

fig, ax = plt.subplots(figsize=(1.5 * n_methods + 2, max(6, n_animals * 0.38)))

colour_map = {"BE": BE_COL, "SC": SC_COL}

for i, (_, row) in enumerate(assign_df.iterrows()):
    for j, col in enumerate(all_cols):
        val = row.get(col)
        p_val = row.get(f"{col}_p", np.nan) if col != "Consensus" else np.nan

        if val in ("BE", "SC"):
            fc = colour_map[val]
            if col == "Consensus":
                alpha = 0.9
            elif pd.notna(p_val) and p_val < 0.05:
                alpha = 0.9
            else:
                alpha = 0.25
            rect = plt.Rectangle((j - 0.45, i - 0.45), 0.9, 0.9,
                                 facecolor=fc, alpha=alpha,
                                 edgecolor="black", lw=0.5)
            ax.add_patch(rect)
            label = val
            if col != "Consensus" and pd.notna(p_val):
                label += f"{p_val:.3f}"
            ax.text(j, i, label, ha="center", va="center",
                    fontsize=7, fontweight="bold",
                    color="white" if alpha > 0.5 else "black")
        elif val == "Split":
            rect = plt.Rectangle((j - 0.45, i - 0.45), 0.9, 0.9,
                                 facecolor="#FFE0B2", edgecolor="black", lw=0.5)
            ax.add_patch(rect)
            ax.text(j, i, "Split", ha="center", va="center",
                    fontsize=7, fontweight="bold", color="#E65100")
        elif val == "Unclear":
            rect = plt.Rectangle((j - 0.45, i - 0.45), 0.9, 0.9,
                                 facecolor="#F0F0F0", edgecolor="#CCC", lw=0.5)
            ax.add_patch(rect)
            ax.text(j, i, "?", ha="center", va="center",
                    fontsize=9, color="#999")
        elif isinstance(val, str) and "inconclusive" in val.lower():
            rect = plt.Rectangle((j - 0.45, i - 0.45), 0.9, 0.9,
                                 facecolor="#D4D4D4", edgecolor="#CCC", lw=0.5)
            ax.add_patch(rect)
            ax.text(j, i, "?", ha="center", va="center",
                    fontsize=9, fontweight="bold", color="#666")
        else:
            rect = plt.Rectangle((j - 0.45, i - 0.45), 0.9, 0.9,
                                 facecolor="#F8F8F8", edgecolor="#DDD", lw=0.5)
            ax.add_patch(rect)
            ax.text(j, i, "—", ha="center", va="center",
                    fontsize=9, color="#BBB")

ax.axvline(n_methods - 1.5, color="black", lw=2.5)
ax.set_xlim(-0.6, n_methods - 0.4)
ax.set_ylim(-0.6, n_animals - 0.4)
ax.set_xticks(range(n_methods))
ax.set_xticklabels(all_cols, fontsize=10, fontweight="bold")
ax.xaxis.set_ticks_position("top")
ax.set_yticks(range(n_animals))
ax.set_yticklabels(assign_df["id"].values, fontsize=8)
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=BE_COL, alpha=0.9, label="BE (p<0.05)"),
    Patch(facecolor=BE_COL, alpha=0.25, label="BE (not sig.)"),
    Patch(facecolor=SC_COL, alpha=0.9, label="SC (p<0.05)"),
    Patch(facecolor=SC_COL, alpha=0.25, label="SC (not sig.)"),
    Patch(facecolor="#FFE0B2", edgecolor="#E65100", label="Split"),
    Patch(facecolor="#D4D4D4", label="Inconclusive"),
    Patch(facecolor="#F0F0F0", label="No data"),
]
ax.legend(handles=legend_elements, loc="upper left",
          bbox_to_anchor=(1.02, 1), fontsize=8)

cons_counts = assign_df["Consensus"].value_counts()
cons_str = ", ".join(f"{k}: {v}" for k, v in cons_counts.items())
ax.set_title(f"Model Assignment — All Animals {cons_str}",
             fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

print(assign_df[["id"] + method_cols + ["Consensus"]].to_string(index=False))

---
## 2. Parameter Distributions Across Cohort

SBI posterior medians + 90% CIs for all animals.
Uses saved posterior samples (fast path) if available,
otherwise re-conditions SNPE.

In [ ]:
if len(snpe) == 2:
    param_data = []

    for aid in sorted(experiment.animals.keys()):
        sessions = select_sessions(experiment.animals[aid], "expert_uniform")
        if len(sessions) < 3:
            continue

        consensus = assign_df.loc[assign_df["id"] == aid, "Consensus"].values
        consensus = consensus[0] if len(consensus) > 0 else "Unclear"

        # Use _get_sbi_samples: loads saved samples or re-conditions
        be_samples, sc_samples, _ = _get_sbi_samples(
            aid, sessions, RESULTS_DIR, snpe, "uniform", n_samples=5000)

        for mk, samples in [("be", be_samples), ("sc", sc_samples)]:
            if samples is None:
                continue
            pnames = snpe[mk]["param_names"]
            for j, pn in enumerate(pnames):
                param_data.append({
                    "animal_id": aid,
                    "model": mk.upper(),
                    "param": pn,
                    "median": np.median(samples[:, j]),
                    "ci_lo": np.percentile(samples[:, j], 5),
                    "ci_hi": np.percentile(samples[:, j], 95),
                    "consensus": consensus,
                })

    param_df = pd.DataFrame(param_data)
    print(f"{param_df['animal_id'].nunique()} animals with SBI parameters")
else:
    param_df = pd.DataFrame()
    print("SNPE networks not loaded — skipping parameter distributions")

In [ ]:
if len(param_df) > 0:
    for mt in ["BE", "SC"]:
        sub = param_df[param_df["model"] == mt]
        params = sub["param"].unique()
        n = len(params)

        fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
        axes = np.atleast_1d(axes)

        for i, pn in enumerate(params):
            ax = axes[i]
            psub = sub[sub["param"] == pn].sort_values("animal_id")

            for j, (_, row) in enumerate(psub.iterrows()):
                cons = row["consensus"]
                col = BE_COL if cons == "BE" else SC_COL if cons == "SC" else "grey"
                ax.errorbar(j, row["median"],
                            yerr=[[row["median"] - row["ci_lo"]],
                                  [row["ci_hi"] - row["median"]]],
                            fmt="o", color=col, ms=5, capsize=3,
                            elinewidth=1, alpha=0.8)

            ax.set_xticks(range(len(psub)))
            ax.set_xticklabels(psub["animal_id"].values,
                               rotation=45, ha="right", fontsize=7)
            ax.set_ylabel(pn)
            ax.set_title(pn, fontsize=11)

        fig.suptitle(f"{mt} — SBI Parameter Estimates (colour = consensus)",
                     fontsize=13, fontweight="bold")
        plt.tight_layout(); plt.show()

---
## 3. GS vs SBI Consistency

For animals with both methods: do the parameter estimates agree?

In [ ]:
from plotting.animal_report import _load_gs_raw_pickles, _gs_seed_errors

consistency_data = {"BE": {}, "SC": {}}

for aid in sorted(experiment.animals.keys()):
    sessions = select_sessions(experiment.animals[aid], "expert_uniform")
    if len(sessions) < 3:
        continue

    be_data, sc_data = _load_gs_raw_pickles(aid, RESULTS_DIR, "uniform", "update_matrix")
    if not be_data or not sc_data:
        continue

    be_samples, sc_samples, _ = _get_sbi_samples(
        aid, sessions, RESULTS_DIR, snpe, "uniform", n_samples=5000)

    for model_name, gs_data, samples in [
        ("BE", be_data, be_samples), ("SC", sc_data, sc_samples)
    ]:
        _, gs_params = _gs_seed_errors(gs_data)
        if not gs_params or samples is None:
            continue

        mk = model_name.lower()
        if mk not in snpe:
            continue
        pnames = snpe[mk]["param_names"]
        for j, pn in enumerate(pnames):
            if pn not in gs_params:
                continue
            if pn not in consistency_data[model_name]:
                consistency_data[model_name][pn] = {"gs": [], "sbi": [], "aid": []}
            consistency_data[model_name][pn]["gs"].append(gs_params[pn])
            consistency_data[model_name][pn]["sbi"].append(np.median(samples[:, j]))
            consistency_data[model_name][pn]["aid"].append(aid)

In [ ]:
for mt in ["BE", "SC"]:
    params = consistency_data.get(mt, {})
    if not params:
        continue
    pnames = list(params.keys())
    n = len(pnames)
    col = BE_COL if mt == "BE" else SC_COL

    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    axes = np.atleast_1d(axes)

    for i, pn in enumerate(pnames):
        ax = axes[i]
        gs_vals = np.array(params[pn]["gs"])
        sbi_vals = np.array(params[pn]["sbi"])
        aids = params[pn]["aid"]

        ax.scatter(gs_vals, sbi_vals, color=col, s=40, alpha=0.7,
                   edgecolors="black", lw=0.5)
        for g, s, a in zip(gs_vals, sbi_vals, aids):
            ax.annotate(a, (g, s), fontsize=6, alpha=0.6,
                        xytext=(3, 3), textcoords="offset points")

        all_v = np.concatenate([gs_vals, sbi_vals])
        lo, hi = all_v.min(), all_v.max()
        m = (hi - lo) * 0.1
        lims = [lo - m, hi + m]
        ax.plot(lims, lims, "k--", alpha=0.3)

        r = np.corrcoef(gs_vals, sbi_vals)[0, 1] if len(gs_vals) > 2 else np.nan
        ax.text(0.05, 0.95, f"r = {r:.3f} n = {len(gs_vals)}",
                transform=ax.transAxes, fontsize=9, va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

        ax.set(xlabel=f"GS-UM {pn}", ylabel=f"SBI {pn}", aspect="equal")
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_title(pn, fontsize=11)

    fig.suptitle(f"{mt} — GS vs SBI Parameter Consistency",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

---
## 4. Example Animal Deep-Dives

Change  at the top to pick different animals.

In [ ]:
for AID in EXAMPLE_ANIMALS:
    print(f"{'═' * 70}")
    print(f"  {AID}")
    print(f"{'═' * 70}")

    if AID not in experiment.animals:
        print(f"{AID} not found"); continue

    sessions = select_sessions(experiment.animals[AID], "expert_uniform")
    if not sessions:
        print("No expert uniform sessions"); continue

    plot_animal_summary(AID, sessions, RESULTS_DIR)

    for ft in FIT_TARGETS:
        plot_cv_results(AID, RESULTS_DIR, fit_target=ft)

    plot_model_fits(AID, sessions, RESULTS_DIR, method="sbi")
    plot_model_fits(AID, sessions, RESULTS_DIR, method="gs")
    plot_sbi_diagnostics(AID, sessions, snpe, RESULTS_DIR)

---
## 5. Dynamic SBI — Parameter Trajectories

How do parameters evolve across expert sessions?
(Fitted to expert uniform sessions only.)

In [ ]:
from plotting.sbi import plot_parameter_trajectories

DYN_DIR = SBI_DYNAMIC_DIR / "uniform_update_matrix"

if DYN_DIR.exists():
    dyn_files = [f for f in sorted(DYN_DIR.glob("*.pkl"))
                 if f.name != "summary.pkl"]

    for AID in EXAMPLE_ANIMALS:
        for model in ["BE", "SC"]:
            p = DYN_DIR / f"{AID}_{model}.pkl"
            if not p.exists():
                continue
            with open(p, "rb") as f:
                d = pickle.load(f)

            varying = d["varying_params"]
            print(f"{AID} {model} ({d['n_sessions']} sessions, "
                  f"varying: {varying})")

            for pn, traj in d["trajectories"].items():
                med = traj["median"]
                if isinstance(med, np.ndarray) and len(med) > 1:
                    print(f"  {pn:20s} {med[0]:.3f} → {med[-1]:.3f} "
                          f"(Δ={med[-1]-med[0]:+.3f})")
                else:
                    val = med if np.isscalar(med) else med[0]
                    print(f"  {pn:20s} constant = {val:.3f}")

            fig = plot_parameter_trajectories(
                d["trajectories"],
                title=f"{AID} — {model} Parameter Trajectories "
                      f"(Dynamic SBI, {d['n_sessions']} sessions)",
                show_samples=20,
            )
            plt.show()
else:
    print("No dynamic SBI results found")

---
## 6. Drill-Down: Any Animal

Change  and re-run this cell.

In [ ]:
INSPECT_ID = "SS04"  # ← change this

if INSPECT_ID not in experiment.animals:
    print(f"{INSPECT_ID} not found. Available:")
    for aid in sorted(experiment.animals.keys()):
        n = len(select_sessions(experiment.animals[aid], "expert_uniform"))
        cons = assign_df.loc[assign_df["id"] == aid, "Consensus"].values
        cons = cons[0] if len(cons) > 0 else "?"
        print(f"  {aid}: {n} expert sessions, consensus={cons}")
else:
    sessions = select_sessions(experiment.animals[INSPECT_ID], "expert_uniform")
    if not sessions:
        print(f"{INSPECT_ID}: no expert sessions")
    else:
        print(f"Inspecting {INSPECT_ID} ({len(sessions)} expert sessions)")

        plot_animal_summary(INSPECT_ID, sessions, RESULTS_DIR)
        for ft in FIT_TARGETS:
            plot_cv_results(INSPECT_ID, RESULTS_DIR, fit_target=ft)
        plot_model_fits(INSPECT_ID, sessions, RESULTS_DIR, method="sbi")
        plot_model_fits(INSPECT_ID, sessions, RESULTS_DIR, method="gs")
        plot_sbi_diagnostics(INSPECT_ID, sessions, snpe, RESULTS_DIR)

        if DYN_DIR.exists():
            for model in ["BE", "SC"]:
                p = DYN_DIR / f"{INSPECT_ID}_{model}.pkl"
                if not p.exists():
                    continue
                with open(p, "rb") as f:
                    d = pickle.load(f)
                fig = plot_parameter_trajectories(
                    d["trajectories"],
                    title=f"{INSPECT_ID} — {model} Trajectories",
                    show_samples=20)
                plt.show()